# NYC Yellow Taxi Dataset Cleaner
This notebook performs a complete cleaning and preparation workflow for the 2025 NYC Taxi dataset.

In [15]:
#  Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# Import libraries
import pandas as pd
import glob
import os

In [17]:
# Load raw files
raw_path = "/content/drive/MyDrive/NYC taxi/*.parquet"
files = sorted(glob.glob(raw_path))

print("Total raw files found:", len(files))
for file in files:
    print(f"Detected: {file}")

Total raw files found: 12
Detected: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-01.parquet
Detected: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-02.parquet
Detected: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-03.parquet
Detected: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-04.parquet
Detected: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-05.parquet
Detected: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-06.parquet
Detected: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-07.parquet
Detected: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-08.parquet
Detected: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-09.parquet
Detected: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-10.parquet
Detected: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-11.parquet
Detected: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-12.parquet


In [18]:
# Read and merge data
use_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "payment_type",
    "PULocationID",
    "DOLocationID"
]

df_list = []
for file in files:
    print("Loading:", file)
    temp_df = pd.read_parquet(file, columns=use_cols)
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)
display(df.head())

Loading: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-01.parquet
Loading: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-02.parquet
Loading: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-03.parquet
Loading: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-04.parquet
Loading: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-05.parquet
Loading: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-06.parquet
Loading: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-07.parquet
Loading: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-08.parquet
Loading: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-09.parquet
Loading: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-10.parquet
Loading: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-11.parquet
Loading: /content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-12.parquet


,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,payment_type,PULocationID,DOLocationID
0,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,10.0,3.00,18.00,1,229,237
1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,5.1,2.02,12.12,1,236,237
2,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,5.1,2.00,12.10,1,141,141
3,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,7.2,0.00,9.70,2,244,244
4,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,5.8,0.00,8.30,2,244,116


In [19]:
# Basic data check
print("Raw dataset shape:", df.shape)
print("\n--- Column Info ---")
df.info()
print("\n--- Missing Values ---")
print(df.isnull().sum())

Raw dataset shape: (48722602, 10)

--- Column Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48722602 entries, 0 to 48722601
Data columns (total 10 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   tpep_pickup_datetime   datetime64[us]
 1   tpep_dropoff_datetime  datetime64[us]
 2   passenger_count        float64       
 3   trip_distance          float64       
 4   fare_amount            float64       
 5   tip_amount             float64       
 6   total_amount           float64       
 7   payment_type           int64         
 8   PULocationID           int32         
 9   DOLocationID           int32         
dtypes: datetime64[us](2), float64(5), int32(2), int64(1)
memory usage: 3.3 GB

--- Missing Values ---
tpep_pickup_datetime            0
tpep_dropoff_datetime           0
passenger_count          11611894
trip_distance                   0
fare_amount                     0
tip_amount                      0
total_a

In [20]:
# Convert date columns
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

In [21]:
# Create trip duration
df["trip_duration_min"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60

In [ ]:
#  Clean invalid records
try:
    # Removing rows that don't represent a valid trip to prevent 'Garbage In, Garbage Out'
    df_cleaned = df[
        (df["trip_distance"] > 0) &
        (df["fare_amount"] > 0) &
        (df["total_amount"] > 0) &
        (df["passenger_count"] > 0) &
        (df["trip_duration_min"] >= 1) &
        (df["trip_duration_min"] <= 180)
    ].copy()
    print("Cleaning complete. Data is now filtered for high-quality analysis.")
except NameError:
    print("Error: 'df' is not in memory. Please re-run Cell 36375de9 (Read and merge data) first.")

In [ ]:
# Remove duplicates
try:
    df_cleaned = df_cleaned.drop_duplicates()
    print("Duplicates removed successfully.")
except NameError:
    print("Error: 'df_cleaned' is not defined. Please ensure Cell 8 executed successfully first.")

In [ ]:
# Create analysis columns
try:
    df_cleaned["pickup_hour"] = df_cleaned["tpep_pickup_datetime"].dt.hour
    df_cleaned["pickup_day"] = df_cleaned["tpep_pickup_datetime"].dt.day_name()
    df_cleaned["pickup_month"] = df_cleaned["tpep_pickup_datetime"].dt.month
    print("Analysis columns created successfully.")
except NameError:
    print("Error: 'df_cleaned' is not defined. Please ensure Cell 8 (Cleaning) ran successfully first.")

In [ ]:
# Cell 11: Final Cleaning Log & Summary
try:
    print("--- NYC TAXI DATA CLEANING LOG ---")
    print(f"Original raw records: {len(df):,}")
    print(f"Cleaned records:      {len(df_cleaned):,}")
    print(f"Total rows removed:   {len(df) - len(df_cleaned):,}")
    print("\nRationale: Removed outliers, zero-passenger trips, and unrealistic durations.")
    display(df_cleaned.head())
except NameError:
    print("Error: Dataframes not found. Ensure Cells 4 and 8 finished successfully.")

In [ ]:
#  Save cleaned dataset
import os

cleaned_path = "/content/drive/MyDrive/NYC taxi/Cleaned Dataset"
os.makedirs(cleaned_path, exist_ok=True)

try:
    save_name = os.path.join(cleaned_path, "yellow_taxi_cleaned_2025.parquet")
    df_cleaned.to_parquet(save_name, index=False)
    print(f"Cleaned dataset successfully saved to: {save_name}")
except NameError:
    print("Error: 'df_cleaned' variable missing. Run the cleaning cell (Cell 8) first.")

### Why we perform these cleaning steps:

*   **Datetime Conversion**: Standardizing time formats is essential for calculating the `trip_duration` and performing any time-series analysis.
*   **Filtering Logic**: We filter out trips with `0` distance or duration because these are usually system errors or GPS glitches that would skew our statistical averages.
*   **Passenger Validation**: Standard taxis carry between 1 and 6 passengers. Removing `0` or `>6` ensures we are analyzing actual passenger trips rather than system tests.
*   **Deduplication**: This prevents 'double-counting' revenue or trip volume, ensuring our final project report is accurate.